# Задание 7


## 📋 Описание задания
**Цель:** Сравнить 4 разных алгоритма классификации на датасете Wine  
**Алгоритмы:**  
1. KNN (K-Nearest Neighbors) - метод ближайших соседей  
2. SVM (Support Vector Machine) - метод опорных векторов  
3. Decision Tree - дерево решений  
4. Logistic Regression - логистическая регрессия  
**Методы:** GridSearchCV для подбора гиперпараметров каждого алгоритма  
**Метрика:** F1-weighted (взвешенная F1-мера)


In [4]:
# ========================================================================
# ИМПОРТ АЛГОРИТМОВ КЛАССИФИКАЦИИ
# ========================================================================
# KNeighborsClassifier - K ближайших соседей (голосование большинства)
# SVC - Support Vector Machine (поиск оптимальной гиперплоскости)
# DecisionTreeClassifier - Дерево решений (последовательность вопросов)
# LogisticRegression - Логистическая регрессия (вероятностная модель)
# ========================================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [5]:
# ========================================================================
# СОЗДАНИЕ PIPELINES ДЛЯ КАЖДОГО АЛГОРИТМА
# ========================================================================
# pipeline_KNN: StandardScaler + KNN
#   - Требует нормализации (работает с расстояниями!)
# pipeline_SVC: StandardScaler + SVM
#   - Требует нормализации (работает с расстояниями!)
# pipeline_DTC: Decision Tree
#   - НЕ требует нормализации (работает с порогами)
# pipeline_Logistic: StandardScaler + LogisticRegression
#   - Требует нормализации (градиентный спуск)
# ========================================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline_KNN = Pipeline([
    ('scaling', StandardScaler()),
    ('regressor', KNeighborsClassifier())
])
pipeline_SVC = Pipeline([
    ('scaling', StandardScaler()),
    ('regressor', SVC())
])
pipeline_DTC = Pipeline([
    ('regressor', DecisionTreeClassifier())
])
pipeline_Logistic = Pipeline([
    ('scaling', StandardScaler()),
    ('regressor', LogisticRegression())
])

In [8]:
# ========================================================================
# СЕТКИ ГИПЕРПАРАМЕТРОВ ДЛЯ КАЖДОГО АЛГОРИТМА
# ========================================================================
# KNN:
#   n_neighbors - количество соседей для голосования
# SVM:
#   C - жёсткость разделения (обратная регуляризация)
#   kernel - тип ядра:
#     'linear' - линейная граница
#     'rbf' - нелинейная граница (гауссово ядро)
# Decision Tree:
#   max_depth - максимальная глубина дерева
#   min_samples_split - минимум объектов для разделения узла
# Logistic Regression:
#   C, penalty, solver - как в Task_4
# ========================================================================

import numpy as np

knn_params = {'regressor__n_neighbors': [3, 5, 7, 9, 11]}
svm_params = {'regressor__C': [0.1, 1, 10], 'regressor__kernel': ['linear', 'rbf']}
dtc_params = {'regressor__max_depth': [3, 5, 7, None], 'regressor__min_samples_split': [2, 5, 10]}
logistic_params = {'regressor__C': np.logspace(-2, 2),
             'regressor__penalty': ['l1', 'l2'],
             'regressor__solver': ['multinomial', 'saga']}

In [16]:
# ========================================================================
# СОЗДАНИЕ GRIDSEARCHCV ДЛЯ КАЖДОГО АЛГОРИТМА
# ========================================================================
# cv=5 - 5-fold кросс-валидация
# scoring='f1_weighted' - метрика качества:
#   F1-мера, взвешенная по количеству объектов каждого класса
#   Баланс между precision (точность) и recall (полнота)
# ========================================================================

from sklearn.model_selection import GridSearchCV

knn_searcher = GridSearchCV(cv = 5, estimator=pipeline_KNN, param_grid = knn_params, scoring='f1_weighted')
svm_searcher = GridSearchCV(cv = 5, estimator = pipeline_SVC, param_grid=svm_params, scoring='f1_weighted')
dtc_searcher = GridSearchCV(estimator=pipeline_DTC, param_grid = dtc_params, cv = 5, scoring='f1_weighted')
logistic_searcher = GridSearchCV(estimator=pipeline_Logistic, param_grid = logistic_params, cv = 5, scoring='f1_weighted')

In [17]:
# ========================================================================
# ЗАГРУЗКА ДАТАСЕТА WINE
# ========================================================================
# Тот же датасет, что в Task_4
# Сравниваем все алгоритмы на одних и тех же данных
# ========================================================================

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine

wine = load_wine()

X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, test_size=0.25, random_state=42)

In [18]:
import warnings

warnings.filterwarnings("ignore")

knn_searcher.fit(X_train, y_train)
svm_searcher.fit(X_train, y_train)
dtc_searcher.fit(X_train, y_train)
logistic_searcher.fit(X_train, y_train)

,estimator,Pipeline(step...egression())])
,param_grid,"{'regressor__C': array([1.0000...00000000e+02]), 'regressor__penalty': ['l1', 'l2'], 'regressor__solver': ['multinomial', 'saga']}"
,scoring,'f1_weighted'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [21]:
best_knn = knn_searcher.best_estimator_.fit(X_train, y_train)
best_svm = svm_searcher.best_estimator_.fit(X_train, y_train)
best_dtc = dtc_searcher.best_estimator_.fit(X_train, y_train)
best_logistic = logistic_searcher.best_estimator_.fit(X_train, y_train)

In [22]:
import time
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

models = {
    'KNN': best_knn,
    'SVM': best_svm,
    'Decision Tree': best_dtc,
    'Logistic Regression': best_logistic
}

results = pd.DataFrame(columns=['Model', 'Accuracy', 'F1-weighted', 'Training Time (s)', 'Prediction Time (s)'])

for model_name, model in models.items():
    print(f"Оценка модели: {model_name}")

    start_pred_time = time.time()
    y_pred = model.predict(X_test)
    end_pred_time = time.time()
    prediction_time = end_pred_time - start_pred_time

    accuracy = accuracy_score(y_test, y_pred)
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    results = pd.concat([results, pd.DataFrame([{
        'Model': model_name,
        'Accuracy': accuracy,
        'F1-weighted': f1_weighted,
        'Training Time (s)': 'уже обучена',
        'Prediction Time (s)': prediction_time
    }])], ignore_index=True)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-weighted: {f1_weighted:.4f}")
    print(f"Время предсказания: {prediction_time:.4f} секунд")
    print("-" * 50)

print("\n" + "="*80)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ:")
print("="*80)
print(results.to_string(index=False))

Оценка модели: KNN
Accuracy: 0.9556
F1-weighted: 0.9551
Время предсказания: 0.0030 секунд
--------------------------------------------------
Оценка модели: SVM
Accuracy: 0.9778
F1-weighted: 0.9779
Время предсказания: 0.0012 секунд
--------------------------------------------------
Оценка модели: Decision Tree
Accuracy: 0.9333
F1-weighted: 0.9338
Время предсказания: 0.0006 секунд
--------------------------------------------------
Оценка модели: Logistic Regression
Accuracy: 0.9778
F1-weighted: 0.9779
Время предсказания: 0.0007 секунд
--------------------------------------------------

ИТОГОВЫЕ РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ:
              Model  Accuracy  F1-weighted Training Time (s)  Prediction Time (s)
                KNN  0.955556     0.955051       уже обучена             0.002967
                SVM  0.977778     0.977905       уже обучена             0.001202
      Decision Tree  0.933333     0.933827       уже обучена             0.000605
Logistic Regression  0.977778     0.9779

> Какой метод показал наилучшее качество? Какова интерпретируемость и скорость работы каждого?

# Сравнение алгоритмов машинного обучения

## 1. Почему kNN чувствителен к масштабу признаков, а дерево решений — нет?

**kNN чувствителен потому что:**
- Использует расстояния между точками (Евклидово, Манхэттенское)
- Признаки с большим масштабом доминируют в вычислении расстояния
- Без масштабирования признаки в разных единицах измерения несравнимы

**Дерево решений не чувствительно потому что:**
- Разбиения основаны на пороговых значениях отдельных признаков
- Работает с порядковыми сравнениями, а не абсолютными значениями
- Каждый признак рассматривается изолированно при построении дерева

## 2. Как выбор ядра (kernel) влияет на качество SVM?

**Линейное ядро:**
- Подходит для линейно разделимых данных
- Быстрое вычисление, меньше гиперпараметров
- Хорошая интерпретируемость

**RBF ядро (радиальное):**
- Решает нелинейные задачи разделения
- Требует тонкой настройки параметра γ
- Может переобучаться при неправильных параметрах

**Полиномиальное ядро:**
- Улавливает полиномиальные зависимости
- Требует настройки степени полинома
- Вычислительно сложнее линейного

## 3. Почему для дерева решений не требуется масштабирование, но его иногда всё равно включают в pipeline?

**Причины включения в pipeline:**
- **Стандартизация workflow** - единый процесс для всех моделей
- **Ансамбли моделей** - в Random Forest или Gradient Boosting масштабирование может помочь
- **Сравнение моделей** - честное сравнение на одинаково обработанных данных
- **Воспроизводимость** - унификация процесса предобработки

## 4. Какие из рассмотренных методов являются линейными, а какие — нелинейными?

**Линейные методы:**
- Логистическая регрессия (без ядерных преобразований)
- Линейный SVM (с линейным ядром)

**Нелинейные методы:**
- kNN (нелинейные границы решений)
- Дерево решений (кусочно-постоянная функция)
- SVM с нелинейными ядрами (RBF, полиномиальное)

## 5. Какие методы лучше интерпретируются человеком: логистическая регрессия или дерево решений? Почему?

**Дерево решений лучше интерпретируется потому что:**
- Визуальное представление в виде дерева правил
- Прозрачная логика принятия решений
- Легко объяснить бизнес-пользователям
- Можно проследить весь путь предсказания

**Логистическая регрессия:**
- Коэффициенты показывают влияние признаков
- Статистическая интерпретируемость
- Сложнее для не технической аудитории
- Требует понимания вероятностей и логитов

## 6. Почему SVM может работать медленнее других методов на больших данных?

**Вычислительная сложность SVM:**
- Время обучения O(n²) - n³ в худшем случае
- Требует решения задачи квадратичного программирования
- Хранит опорные векторы в памяти
- Не эффективен для datasets с миллионами строк

## 7. Какие методы склонны к переобучению, если не ограничить сложность?

**Склонны к переобучению:**
- Дерево решений (без max_depth, min_samples_split)
- kNN (при малом k, особенно k=1)
- SVM (с неправильно подобранными C и γ)
- Нейронные сети (без регуляризации)

**Устойчивые к переобучению:**
- Логистическая регрессия (с L1/L2 регуляризацией)
- Линейный SVM (с правильной регуляризацией)

## 8. Как метрика f1_weighted помогает объективно сравнить модели при несбалансированных классах?

**f1_weighted:**
- Учитывает количество примеров в каждом классе
- Взвешивает F1-score по поддержке (support) классов
- Не позволяет мажоритарным классам доминировать в оценке
- Дает репрезентативную картину качества по всем классам
- Сравнивает модели на основе их реальной практической полезности

## 9. В каких случаях логистическая регрессия может проигрывать kNN или SVM?

**Логистическая регрессия проигрывает когда:**
- Сильно нелинейные границы между классами
- Сложные взаимодействия между признаками
- Данные имеют сложную геометрию разделения
- Не выполняется предположение о линейной разделимости

## 10. Почему важно использовать одинаковые разбиения данных (train/test) при сравнении моделей?

**Причины использования одинаковых разбиений:**
- **Честное сравнение** - одинаковые условия для всех моделей
- **Устранение случайности** - исключение влияния "удачного" разбиения
- **Воспроизводимость** - возможность повторить эксперимент
- **Статистическая значимость** - корректные статистические тесты
- **Сравнимость метрик** - метрики вычислены на идентичных данных